# ⚡ Notebook 3 — TFLite + TFJS Conversion
**Prerequisite**: Run notebooks 1 and 2 first (in the same session).
Needs: `/content/model/crop_disease_model.h5` and `/content/data/val/`

**Downloads at the end**:
- `tfjs_model_for_app.zip` → unzip into `app/public/model/`
- `model_artifacts.zip` → TFLite + training outputs

In [ ]:
!pip install -q tensorflowjs==4.15.0

In [ ]:
import tensorflow as tf
import numpy as np
import json, os, shutil

print(f'TF: {tf.__version__}')

MODEL_DIR = '/content/model'
DATA_DIR  = '/content/data'
IMG_SIZE  = 224

with open(f'{MODEL_DIR}/labels.json') as f:
    LABELS_MAP = json.load(f)
NUM_CLASSES = len(LABELS_MAP)
print(f'Classes: {NUM_CLASSES} → {LABELS_MAP}')

os.makedirs('/content/tfjs_model', exist_ok=True)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# LOAD TRAINED MODEL
# ─────────────────────────────────────────────────────────────────────────────

MODEL_PATH = f'{MODEL_DIR}/crop_disease_model.h5'

if not os.path.exists(MODEL_PATH):
    print('⚠️  Model not found. Either:')
    print('   1. Run notebook 2 in this same session first')
    print('   2. Upload the .h5 file:')
    print('      from google.colab import files; files.upload()')
else:
    model = tf.keras.models.load_model(MODEL_PATH)
    print(f'✅ Model loaded from {MODEL_PATH}')
    print(f'   Input:  {model.input_shape}')
    print(f'   Output: {model.output_shape}')

In [ ]:
# Uncomment to upload model manually:
# from google.colab import files
# uploaded = files.upload()   # upload crop_disease_model.h5
# import shutil; shutil.move('crop_disease_model.h5', MODEL_PATH)
# model = tf.keras.models.load_model(MODEL_PATH)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# BUILD CALIBRATION DATASET FROM /content/data/val/
# (replaces the old TFDS-based calibration)
# ─────────────────────────────────────────────────────────────────────────────

print('Building calibration dataset from validation images...')

calib_ds = tf.keras.utils.image_dataset_from_directory(
    f'{DATA_DIR}/val',
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=None,
    shuffle=True,
    seed=42,
    class_names=LABELS_MAP,
)

def preprocess_calib(image, label):
    image = tf.keras.applications.mobilenet_v2.preprocess_input(
        tf.cast(image, tf.float32)
    )
    return tf.expand_dims(image, 0)  # add batch dim for TFLite converter

calib_images = list(calib_ds.take(200).map(preprocess_calib))
print(f'Calibration samples: {len(calib_images)}')

def representative_dataset():
    for sample in calib_images:
        yield [sample]

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CONVERT TO TFLITE (INT8 quantized)
# ─────────────────────────────────────────────────────────────────────────────

TFLITE_PATH = f'{MODEL_DIR}/crop_disease_model.tflite'

converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_dataset
converter.target_spec.supported_ops = [
    tf.lite.OpsSet.TFLITE_BUILTINS_INT8,
    tf.lite.OpsSet.TFLITE_BUILTINS,
]
converter.inference_input_type  = tf.int8
converter.inference_output_type = tf.int8

print('Converting to INT8 TFLite (~3-5 min)...')
tflite_model = converter.convert()

with open(TFLITE_PATH, 'wb') as f:
    f.write(tflite_model)

orig_mb   = os.path.getsize(MODEL_PATH) / 1e6
tflite_mb = os.path.getsize(TFLITE_PATH) / 1e6
print(f'\n✅ TFLite conversion done!')
print(f'   .h5 size:     {orig_mb:.1f} MB')
print(f'   .tflite size: {tflite_mb:.1f} MB  ({orig_mb/tflite_mb:.1f}× compression)')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# VERIFY TFLITE ON SAMPLE IMAGES
# ─────────────────────────────────────────────────────────────────────────────

interpreter = tf.lite.Interpreter(model_path=TFLITE_PATH)
interpreter.allocate_tensors()
inp = interpreter.get_input_details()[0]
out = interpreter.get_output_details()[0]

print(f'Input  dtype={inp["dtype"]}, shape={inp["shape"]}, quant={inp["quantization"]}')
print(f'Output dtype={out["dtype"]}, shape={out["shape"]}, quant={out["quantization"]}')

import pathlib
correct = 0; total = 0
print(f'\n{"-"*55}')
print(f'{"True":<25} {"Predicted":<25} {"Conf"}')
print(f'{"-"*55}')

for class_key in LABELS_MAP:
    class_dir = pathlib.Path(f'{DATA_DIR}/val/{class_key}')
    imgs = list(class_dir.glob('*.jpg'))[:2] + list(class_dir.glob('*.JPG'))[:1]
    for img_path in imgs[:1]:  # test 1 per class
        raw = tf.io.read_file(str(img_path))
        img = tf.image.decode_jpeg(raw, channels=3)
        img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
        img = tf.keras.applications.mobilenet_v2.preprocess_input(tf.cast(img, tf.float32))

        # Quantize input
        in_scale, in_zp = inp['quantization']
        img_q = np.clip(img.numpy() / in_scale + in_zp, -128, 127).astype(np.int8)
        interpreter.set_tensor(inp['index'], img_q[np.newaxis])
        interpreter.invoke()
        output = interpreter.get_tensor(out['index'])

        # Dequantize + softmax
        out_scale, out_zp = out['quantization']
        probs = tf.nn.softmax((output.astype(np.float32) - out_zp) * out_scale).numpy()[0]
        pred_idx = np.argmax(probs)
        true_idx = LABELS_MAP.index(class_key)
        match = '✅' if pred_idx == true_idx else '❌'
        print(f'{class_key:<25} {LABELS_MAP[pred_idx]:<25} {probs[pred_idx]:.2f} {match}')
        correct += int(pred_idx == true_idx); total += 1

print(f'\nSample accuracy: {correct}/{total} = {correct/total*100:.1f}%')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CONVERT TO TENSORFLOW.JS (float16, for browser inference)
# ─────────────────────────────────────────────────────────────────────────────

print('Converting to TensorFlow.js format...')
!tensorflowjs_converter \
    --input_format=keras \
    --output_format=tfjs_layers_model \
    --quantize_float16 \
    /content/model/crop_disease_model.h5 \
    /content/tfjs_model/

tfjs_mb = sum(os.path.getsize(os.path.join('/content/tfjs_model', f))
              for f in os.listdir('/content/tfjs_model')) / 1e6

print(f'\nTFJS model size: {tfjs_mb:.1f} MB')
!ls -lh /content/tfjs_model/

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# PACKAGE AND DOWNLOAD
# ─────────────────────────────────────────────────────────────────────────────

from google.colab import files

# Add labels.json into tfjs_model folder (app reads this at startup)
shutil.copy(f'{MODEL_DIR}/labels.json', '/content/tfjs_model/labels.json')

# Zip for download
shutil.make_archive('/content/tfjs_model_for_app', 'zip', '/content/tfjs_model')
shutil.make_archive('/content/model_artifacts',    'zip', '/content/model')

print('\n📦 Instructions after downloading:')
print('  1. Unzip tfjs_model_for_app.zip')
print('  2. Copy ALL contents to:  crop-disease-diagnostician/app/public/model/')
print('  3. cd app && npm run dev')
print()

print('Downloading tfjs_model_for_app.zip...')
files.download('/content/tfjs_model_for_app.zip')

print('Downloading model_artifacts.zip...')
files.download('/content/model_artifacts.zip')

print('\n✅ Notebook 3 complete!')